In [12]:
import random
import math
import heapq


# ---------------------------------------------------------
# CLASE NODO
# ---------------------------------------------------------
class Nodo:
    def __init__(self, id, x=None, y=None):
        self.id = id
        self.x = x
        self.y = y
        self.aristas = []

    def grado(self):
        return len(self.aristas)

    def __repr__(self):
        return f"Nodo({self.id})"


# ---------------------------------------------------------
# CLASE ARISTA
# ---------------------------------------------------------
class Arista:
    def __init__(self, origen, destino, peso=1):
        self.origen = origen
        self.destino = destino
        self.peso = peso

    def __repr__(self):
        return f"{self.origen.id} -> {self.destino.id} (w={self.peso:.4f})"


# ---------------------------------------------------------
# CLASE GRAFO
# ---------------------------------------------------------
class Grafo:
    def __init__(self, dirigido=False):
        self.nodos = {}
        self.aristas = []
        self.dirigido = dirigido

    # -----------------------------------------------------
    # Agregar nodo
    # -----------------------------------------------------
    def agregar_nodo(self, id, x=None, y=None):
        if id not in self.nodos:
            self.nodos[id] = Nodo(id, x, y)

    # -----------------------------------------------------
    # Agregar arista
    # -----------------------------------------------------
    def agregar_arista(self, origen, destino, peso=1):
        if origen == destino:
            return

        n1 = self.nodos[origen]
        n2 = self.nodos[destino]

        arista = Arista(n1, n2, peso)
        self.aristas.append(arista)
        n1.aristas.append(arista)

        if not self.dirigido:
            arista2 = Arista(n2, n1, peso)
            self.aristas.append(arista2)
            n2.aristas.append(arista2)

    # -----------------------------------------------------
    # Devuelve una arista por par (evita duplicados no dirigidos)
    # -----------------------------------------------------
    def _aristas_unicas(self):
        seen = set()
        result = []
        for arista in self.aristas:
            if self.dirigido:
                key = (arista.origen.id, arista.destino.id)
            else:
                key = frozenset([arista.origen.id, arista.destino.id])
            if key not in seen:
                seen.add(key)
                result.append(arista)
        return result

    # -----------------------------------------------------
    # Guardar grafo en formato GEXF (Gephi)
    # -----------------------------------------------------
    def guardar_gexf(self, archivo):
        with open(archivo, "w", encoding="utf-8") as f:
            f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
            f.write('<gexf xmlns="http://www.gexf.net/1.2draft"\n')
            f.write('      xmlns:viz="http://www.gexf.net/1.2draft/viz"\n')
            f.write('      version="1.2">\n')

            defaultedgetype = "directed" if self.dirigido else "undirected"
            f.write(f'  <graph mode="static" defaultedgetype="{defaultedgetype}">\n')

            f.write('    <nodes>\n')
            for nodo in self.nodos.values():
                node_id = str(nodo.id)
                f.write(f'      <node id="{node_id}" label="{node_id}">\n')
                if nodo.x is not None and nodo.y is not None:
                    f.write(f'        <viz:position x="{nodo.x}" y="{nodo.y}" z="0.0"/>\n')
                f.write('      </node>\n')
            f.write('    </nodes>\n')

            f.write('    <edges>\n')
            edge_id = 0
            agregadas = set()

            for arista in self.aristas:
                origen  = str(arista.origen.id)
                destino = str(arista.destino.id)
                peso    = arista.peso

                if self.dirigido:
                    f.write(
                        f'      <edge id="{edge_id}" source="{origen}" target="{destino}" weight="{peso}"/>\n'
                    )
                    edge_id += 1
                else:
                    clave = tuple(sorted([origen, destino]))
                    if clave in agregadas:
                        continue
                    agregadas.add(clave)
                    f.write(
                        f'      <edge id="{edge_id}" source="{origen}" target="{destino}" weight="{peso}"/>\n'
                    )
                    edge_id += 1

            f.write('    </edges>\n')
            f.write('  </graph>\n')
            f.write('</gexf>\n')

    # =========================================================
    # ÁRBOL DE EXPANSIÓN MÍNIMA
    # =========================================================

    # -----------------------------------------------------
    # KRUSKAL DIRECTO
    # Ordena aristas de menor a mayor peso.
    # Agrega cada arista si no forma un ciclo (Union-Find).
    # Complejidad: O(E log E)
    # -----------------------------------------------------
    def KruskalD(self):
        parent = {nid: nid for nid in self.nodos}
        rank   = {nid: 0   for nid in self.nodos}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]  # path halving
                x = parent[x]
            return x

        def union(x, y):
            px, py = find(x), find(y)
            if px == py:
                return False
            if rank[px] < rank[py]:
                px, py = py, px
            parent[py] = px
            if rank[px] == rank[py]:
                rank[px] += 1
            return True

        aristas = sorted(self._aristas_unicas(), key=lambda a: a.peso)

        mst = Grafo(self.dirigido)
        for nid, nodo in self.nodos.items():
            mst.agregar_nodo(nid, nodo.x, nodo.y)

        count = 0
        for arista in aristas:
            u, v = arista.origen.id, arista.destino.id
            if union(u, v):
                mst.agregar_arista(u, v, arista.peso)
                count += 1
                if count == len(self.nodos) - 1:
                    break

        return mst

    # -----------------------------------------------------
    # KRUSKAL INVERSO
    # Ordena aristas de mayor a menor peso.
    # Elimina la arista si el grafo sigue siendo conexo
    # (la arista no es un puente).
    # Complejidad: O(E^2)  — O(E * (V+E)) por cada DFS
    # -----------------------------------------------------
    def KruskalI(self):
        # Adyacencia mutable: adj[u][v] = peso
        adj = {nid: {} for nid in self.nodos}
        for arista in self._aristas_unicas():
            u, v, p = arista.origen.id, arista.destino.id, arista.peso
            adj[u][v] = p
            adj[v][u] = p

        def es_conexo_sin(eu, ev):
            """DFS que ignora la arista (eu, ev)."""
            inicio = next(iter(adj))
            visitados = set()
            pila = [inicio]
            while pila:
                nodo = pila.pop()
                if nodo in visitados:
                    continue
                visitados.add(nodo)
                for vecino in adj[nodo]:
                    if (nodo == eu and vecino == ev) or (nodo == ev and vecino == eu):
                        continue
                    if vecino not in visitados:
                        pila.append(vecino)
            return len(visitados) == len(adj)

        aristas = sorted(self._aristas_unicas(), key=lambda a: a.peso, reverse=True)

        for arista in aristas:
            u, v = arista.origen.id, arista.destino.id
            if v in adj.get(u, {}):           # arista todavía existe
                if es_conexo_sin(u, v):       # no es puente → se elimina
                    del adj[u][v]
                    del adj[v][u]

        # Construir MST con las aristas restantes
        mst = Grafo(self.dirigido)
        for nid, nodo in self.nodos.items():
            mst.agregar_nodo(nid, nodo.x, nodo.y)

        agregadas = set()
        for u in adj:
            for v, peso in adj[u].items():
                key = frozenset([u, v])
                if key not in agregadas:
                    agregadas.add(key)
                    mst.agregar_arista(u, v, peso)

        return mst

    # -----------------------------------------------------
    # PRIM
    # Crece el MST desde un nodo inicial con un min-heap.
    # Siempre elige la arista de menor peso que conecta
    # un nodo visitado con uno no visitado.
    # Complejidad: O(E log E)
    # -----------------------------------------------------
    def Prim(self, inicio=None):
        if not self.nodos:
            return Grafo(self.dirigido)

        if inicio is None:
            inicio = next(iter(self.nodos))

        mst = Grafo(self.dirigido)
        for nid, nodo in self.nodos.items():
            mst.agregar_nodo(nid, nodo.x, nodo.y)

        visitados = {inicio}
        # (peso, contador, arista) — contador como desempate para evitar
        # comparar objetos Arista entre sí
        heap = []
        contador = 0

        for arista in self.nodos[inicio].aristas:
            heapq.heappush(heap, (arista.peso, contador, arista))
            contador += 1

        while heap and len(visitados) < len(self.nodos):
            peso, _, arista = heapq.heappop(heap)
            u, v = arista.origen.id, arista.destino.id

            if v in visitados:
                continue

            visitados.add(v)
            mst.agregar_arista(u, v, peso)

            for sig in self.nodos[v].aristas:
                if sig.destino.id not in visitados:
                    heapq.heappush(heap, (sig.peso, contador, sig))
                    contador += 1

        return mst


# =========================================================
# MODELOS DE GENERACIÓN DE GRAFOS
# =========================================================

def grafoMalla(m, n, dirigido=False):
    g = Grafo(dirigido)
    for i in range(m):
        for j in range(n):
            g.agregar_nodo((i, j))
    for i in range(m):
        for j in range(n):
            if i < m - 1:
                g.agregar_arista((i, j), (i + 1, j))
            if j < n - 1:
                g.agregar_arista((i, j), (i, j + 1))
    return g


def grafoErdosRenyi(n, m, dirigido=False):
    g = Grafo(dirigido)
    for i in range(n):
        g.agregar_nodo(i)
    posibles = [(i, j) for i in range(n) for j in range(n) if i != j]
    aristas = random.sample(posibles, m)
    for (i, j) in aristas:
        g.agregar_arista(i, j)
    return g


def grafoGilbert(n, p, dirigido=False):
    g = Grafo(dirigido)
    for i in range(n):
        g.agregar_nodo(i)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if not dirigido and j <= i:
                continue
            if random.random() <= p:
                g.agregar_arista(i, j)
    return g


def grafoGeografico(n, r, dirigido=False):
    g = Grafo(dirigido)
    for i in range(n):
        x = random.random()
        y = random.random()
        g.agregar_nodo(i, x, y)
    nodos = list(g.nodos.values())
    for i in range(n):
        for j in range(i + 1, n):
            dx = nodos[i].x - nodos[j].x
            dy = nodos[i].y - nodos[j].y
            dist = math.sqrt(dx**2 + dy**2)
            if dist <= r:
                # La distancia euclidiana es el peso de la arista
                g.agregar_arista(nodos[i].id, nodos[j].id, round(dist, 6))
    return g


def grafoBarabasiAlbert(n, d, dirigido=False):
    g = Grafo(dirigido)
    for i in range(d):
        g.agregar_nodo(i)
    for i in range(d):
        for j in range(i + 1, d):
            g.agregar_arista(i, j)
    for nuevo in range(d, n):
        g.agregar_nodo(nuevo)
        nodos_existentes = list(g.nodos.values())
        grados = [nodo.grado() for nodo in nodos_existentes]
        suma_grados = sum(grados)
        conectados = set()
        while len(conectados) < d:
            r = random.uniform(0, suma_grados)
            acumulado = 0
            for nodo in nodos_existentes:
                acumulado += nodo.grado()
                if acumulado >= r:
                    conectados.add(nodo.id)
                    break
        for v in conectados:
            g.agregar_arista(nuevo, v)
    return g


def grafoDorogovtsevMendes(n, dirigido=False):
    g = Grafo(dirigido)
    for i in range(3):
        g.agregar_nodo(i)
    g.agregar_arista(0, 1)
    g.agregar_arista(1, 2)
    g.agregar_arista(2, 0)
    for nuevo in range(3, n):
        g.agregar_nodo(nuevo)
        arista = random.choice(g.aristas)
        u = arista.origen.id
        v = arista.destino.id
        g.agregar_arista(nuevo, u)
        g.agregar_arista(nuevo, v)
    return g


In [15]:
nodos = 500
g1 = grafoMalla(10,10)

# Aristas para Erdos-Renyi: ≈ n·ln(n)*2 para garantizar conexidad,
# acotado por el máximo posible n·(n-1)
erdos_m = min(int(nodos * math.log(nodos) * 2), nodos * (nodos - 1))
g2 = grafoErdosRenyi(nodos, erdos_m)
g3 = grafoGilbert(nodos, 0.05)
g4 = grafoGeografico(nodos, 0.2)
g5 = grafoBarabasiAlbert(nodos, 3)
g6 = grafoDorogovtsevMendes(nodos)

n = str(nodos)

g1.guardar_gexf("malla"    + n + ".gexf")
g2.guardar_gexf("erdos"    + n + ".gexf")
g3.guardar_gexf("gilbert"  + n + ".gexf")
g4.guardar_gexf("geo"      + n + ".gexf")
g5.guardar_gexf("barabasi" + n + ".gexf")
g6.guardar_gexf("dm"       + n + ".gexf")

# --- Kruskal Directo ---
g1.KruskalD().guardar_gexf("malla"    + n + "_kruskalD.gexf")
g2.KruskalD().guardar_gexf("erdos"    + n + "_kruskalD.gexf")
g3.KruskalD().guardar_gexf("gilbert"  + n + "_kruskalD.gexf")
g4.KruskalD().guardar_gexf("geo"      + n + "_kruskalD.gexf")
g5.KruskalD().guardar_gexf("barabasi" + n + "_kruskalD.gexf")
g6.KruskalD().guardar_gexf("dm"       + n + "_kruskalD.gexf")

# --- Kruskal Inverso ---
g1.KruskalI().guardar_gexf("malla"    + n + "_kruskalI.gexf")
g2.KruskalI().guardar_gexf("erdos"    + n + "_kruskalI.gexf")
g3.KruskalI().guardar_gexf("gilbert"  + n + "_kruskalI.gexf")
g4.KruskalI().guardar_gexf("geo"      + n + "_kruskalI.gexf")
g5.KruskalI().guardar_gexf("barabasi" + n + "_kruskalI.gexf")
g6.KruskalI().guardar_gexf("dm"       + n + "_kruskalI.gexf")

# --- Prim ---
g1.Prim().guardar_gexf("malla"    + n + "_prim.gexf")
g2.Prim().guardar_gexf("erdos"    + n + "_prim.gexf")
g3.Prim().guardar_gexf("gilbert"  + n + "_prim.gexf")
g4.Prim().guardar_gexf("geo"      + n + "_prim.gexf")
g5.Prim().guardar_gexf("barabasi" + n + "_prim.gexf")
g6.Prim().guardar_gexf("dm"       + n + "_prim.gexf")


In [14]:
# =========================================================
# PRUEBA DE ALGORITMOS MST
# =========================================================

# Grafo pequeño para verificar resultados a mano
g_test = grafoGeografico(10, 0.6)

mst_kd = g_test.KruskalD()
mst_ki = g_test.KruskalI()
mst_p  = g_test.Prim()

n = len(g_test.nodos)
print(f"Nodos: {n}  |  Aristas originales: {len(g_test._aristas_unicas())}")
print(f"KruskalD  → aristas en MST: {len(mst_kd._aristas_unicas())}  (esperado {n-1})")
print(f"KruskalI  → aristas en MST: {len(mst_ki._aristas_unicas())}  (esperado {n-1})")
print(f"Prim      → aristas en MST: {len(mst_p._aristas_unicas())}   (esperado {n-1})")

peso_kd = sum(a.peso for a in mst_kd._aristas_unicas())
peso_ki = sum(a.peso for a in mst_ki._aristas_unicas())
peso_p  = sum(a.peso for a in mst_p._aristas_unicas())
print(f"\nPeso total KruskalD : {peso_kd:.6f}")
print(f"Peso total KruskalI : {peso_ki:.6f}")
print(f"Peso total Prim     : {peso_p:.6f}")


Nodos: 10  |  Aristas originales: 35
KruskalD  → aristas en MST: 9  (esperado 9)
KruskalI  → aristas en MST: 9  (esperado 9)
Prim      → aristas en MST: 9   (esperado 9)

Peso total KruskalD : 1.538913
Peso total KruskalI : 1.538913
Peso total Prim     : 1.538913


In [ ]:
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, ast

MST_SUFIJOS = ('_kruskalD', '_kruskalI', '_prim')

def base_de_mst(nombre):
    """Si el archivo es un MST, retorna el nombre del grafo original; si no, None."""
    stem = nombre[:-5]  # quitar .gexf
    for s in MST_SUFIJOS:
        if stem.endswith(s):
            return stem[:-len(s)] + '.gexf'
    return None

def get_pos(G):
    """
    Extrae posiciones de los nodos. Prioridad:
      1. viz:position del GEXF  (grafos geográficos)
      2. IDs con forma de tupla  (grafos malla)
      3. circular layout         (resto — no requiere scipy)
    """
    pos = {}
    for node, data in G.nodes(data=True):
        try:
            p = data['viz']['position']
            pos[node] = (float(p['x']), float(p['y']))
            continue
        except (KeyError, TypeError):
            pass
        try:
            t = ast.literal_eval(node)
            if isinstance(t, tuple) and len(t) == 2:
                pos[node] = (float(t[1]), -float(t[0]))
        except (ValueError, SyntaxError):
            pass
    if len(pos) < len(G.nodes()):
        pos = nx.circular_layout(G)
    return pos

archivos_gexf = sorted(f for f in os.listdir('.') if f.endswith('.gexf'))

for archivo in archivos_gexf:
    base = base_de_mst(archivo)
    fig, ax = plt.subplots(figsize=(10, 8))
    G = nx.read_gexf(archivo)

    if base and os.path.exists(base):
        # ── Archivo MST: grafo original (gris) + aristas MST encima (flechas rojas) ──
        G_orig = nx.read_gexf(base)
        pos = get_pos(G_orig)

        # Aristas y nodos del grafo original
        nx.draw_networkx_edges(G_orig, pos, ax=ax,
                               edge_color='#cccccc', width=0.8, arrows=False)
        nx.draw_networkx_nodes(G_orig, pos, ax=ax,
                               node_color='#aec6e8', node_size=40, linewidths=0)

        # Aristas MST como flechas rojas
        nx.draw_networkx_edges(G, pos, ax=ax,
                               edge_color='#e63946', width=2.0,
                               arrows=True, arrowstyle='-|>',
                               arrowsize=12, connectionstyle='arc3,rad=0.08')

        # Nodos del MST (naranja) encima
        nx.draw_networkx_nodes(G, pos, ax=ax,
                               node_color='#f4a261', node_size=70,
                               linewidths=0.5, edgecolors='#e63946')

        leyenda = [
            mpatches.Patch(color='#aec6e8', label='Nodos originales'),
            mpatches.Patch(color='#f4a261', label='Nodos MST'),
            mpatches.Patch(color='#cccccc', label='Aristas originales'),
            mpatches.Patch(color='#e63946', label='Aristas MST'),
        ]
        ax.legend(handles=leyenda, loc='upper right', fontsize=8)

    else:
        # ── Grafo original ──
        pos = get_pos(G)
        nx.draw_networkx_edges(G, pos, ax=ax,
                               edge_color='#888888', width=0.8, arrows=False)
        nx.draw_networkx_nodes(G, pos, ax=ax,
                               node_color='#aec6e8', node_size=40, linewidths=0)

    ax.set_title(archivo[:-5], fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    salida = archivo[:-5] + '.png'
    plt.savefig(salida, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Guardado: {salida}')


In [ ]:
import shutil, os

modelos = ['malla', 'erdos', 'gilbert', 'geo', 'barabasi', 'dm']

for modelo in modelos:
    os.makedirs(modelo, exist_ok=True)

for f in sorted(os.listdir('.')):
    if not (f.endswith('.gexf') or f.endswith('.png')):
        continue
    for modelo in modelos:
        if f.startswith(modelo):
            destino = os.path.join(modelo, f)
            shutil.move(f, destino)
            print(f'{f} → {modelo}/')
            break
